# RAG (Retrieval-Augmented Generation) - как дать LLM доступ к вашим данным

**Роль:** Преподаватель по ML
**Уровень:** средний (основы Python + LLM)
**Время прохождения:** ~120-150 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **почему** LLM галлюцинируют и как RAG решает эту проблему;
- изучите **архитектуру RAG** от документов до ответа;
- реализуете **три стратегии чанкинга** и сравните их;
- создадите **эмбеддинги** с помощью sentence-transformers;
- построите **векторную базу FAISS** с нуля;
- соберёте **полный RAG-пайплайн** от загрузки до генерации;
- запустите **FastAPI сервер** для RAG-запросов;
- исследуете параметры RAG через **интерактивные виджеты**;
- оцените качество RAG через **precision@k, recall, MRR**;
- реализуете **гибридный поиск** (BM25 + семантический) и ре-ранжирование;
- сравните **RAG vs fine-tuning vs prompting**.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Почему RAG? | Галлюцинации, устаревшие знания, цена fine-tuning |
| 3 | Архитектура RAG | Пайплайн: retrieve -> generate |
| 4 | Стратегии чанкинга | Фиксированный, по предложениям, семантический |
| 5 | Эмбеддинги документов | sentence-transformers на практике |
| 6 | Векторная база: FAISS с нуля | Создание индекса, поиск, сходство |
| 7 | Простой RAG-пайплайн | Загрузка -> чанкинг -> эмбеддинг -> поиск -> генерация |
| 8 | FastAPI сервер для RAG | Сервер с эндпоинтами /query, /add_document, /search |
| 9 | Интерактивный RAG-эксплорер | Виджеты: chunk_size, top_k, temperature, similarity_threshold |
| 10 | Оценка качества RAG | precision@k, recall, MRR |
| 11 | Продвинутый: ре-ранжирование и гибридный поиск | BM25 + семантический поиск |
| 12 | RAG vs fine-tuning vs prompting | Сравнительная таблица и диаграмма |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **sentence-transformers** | Создание эмбеддингов предложений |
| **faiss-cpu** | Векторный поиск и индексация |
| **numpy** | Математические операции и массивы |
| **matplotlib** | Визуализация: графики, тепловые карты, диаграммы |
| **ipywidgets** | Интерактивные виджеты для исследования RAG |
| **fastapi** | Создание сервера для RAG-запросов |
| **uvicorn** | ASGI-сервер для запуска FastAPI |
| **pyngrok** | Публичный доступ к серверу из Colab |
| **rank-bm25** | BM25 алгоритм для гибридного поиска |
| **sklearn** | t-SNE визуализация и метрики |

In [ ]:
# ============================================================
# ШАГ 1: Устанавливаем необходимые библиотеки
# ============================================================

!pip install -q sentence-transformers faiss-cpu pyngrok rank-bm25 scikit-learn  # устанавливаем библиотеки
!pip install -q fastapi uvicorn pydantic                                        # серверные библиотеки

print("Все библиотеки установлены!")

In [ ]:
# ============================================================
# ШАГ 1 (продолжение): Импортируем все нужные модули
# ============================================================

import numpy as np                                  # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                     # для построения графиков и визуализаций
import json                                         # для работы с JSON форматом
import os                                           # для работы с файловой системой
import re                                           # регулярные выражения для обработки текста
import math                                         # математические функции
from collections import defaultdict                 # словарь со значениями по умолчанию
from IPython.display import display, HTML, clear_output  # красивый вывод в ноутбуке
import ipywidgets as widgets                        # интерактивные виджеты
from ipywidgets import interact, interactive, fixed, Layout  # декораторы и_LAYOUT для виджетов

# Настройка matplotlib для русского текста
plt.rcParams['figure.figsize'] = (12, 7)           # размер графиков по умолчанию
plt.rcParams['font.size'] = 12                     # размер шрифта
plt.rcParams['axes.grid'] = True                   # показываем сетку

# Фиксируем random seed для воспроизводимости
np.random.seed(42)                                 # seed для NumPy

print("Все модули импортированы!")

In [ ]:
# ============================================================
# ШАГ 1 (продолжение): Импортируем ML-библиотеки
# ============================================================

from sentence_transformers import SentenceTransformer  # модель для эмбеддингов предложений
import faiss                                           # библиотека векторного поиска от Facebook
from rank_bm25 import BM25Okapi                        # BM25 алгоритм для ключевых слов
from sklearn.manifold import TSNE                      # t-SNE для визуализации эмбеддингов
from sklearn.metrics.pairwise import cosine_similarity  # косинусное сходство

print("ML-библиотеки готовы!")

---
## Шаг 2. Почему RAG? - галлюцинации, устаревшие знания, цена fine-tuning

Большие языковые модели (LLM) впечатляют, но у них есть три фундаментальные проблемы:

### Проблема 1: Галлюцинации
LLM генерируют **правдоподобные, но неверные** ответы. Модель "придумывает" факты, потому что она не имеет доступа к источникам.

### Проблема 2: Устаревшие знания
Обучающие данные LLM имеют **дату отсечки**. Модель не знает о событиях после обучения.

### Проблема 3: Цена fine-tuning
Дообучение модели на новых данных - **дорого и медленно**. Нужно обновлять при каждом изменении.

### Решение: RAG
RAG (Retrieval-Augmented Generation) решает все три проблемы:
1. **Находит** релевантные документы из базы знаний
2. **Передаёт** их как контекст в LLM
3. LLM **генерирует** ответ на основе реальных данных

Вместо запоминания всего - поиск по мере необходимости!

In [ ]:
# ============================================================
# ШАГ 2: Демонстрация проблемы галлюцинаций
# ============================================================

# Примеры вопросов, на которых LLM может галлюцинировать
# Показываем, почему нужен доступ к актуальным данным

hallucination_examples = [                             # список примеров галлюцинаций
    {                                                 # пример 1
        "question": "Кто выиграл Нобелевскую премию по физике в 2024?",  # вопрос
        "without_rag": "Модель может назвать случайного учёного или старого лауреата",  # без RAG
        "with_rag": "Модель найдёт точный ответ из свежих источников"   # с RAG
    },
    {                                                 # пример 2
        "question": "Какой API у новой версии библиотеки X?",  # вопрос
        "without_rag": "Модель может описать старый API или выдумать несуществующие методы",  # без RAG
        "with_rag": "Модель найдёт актуальную документацию и даст точный ответ"  # с RAG
    },
    {                                                 # пример 3
        "question": "Какова политика компании Y по удалённой работе?",  # вопрос
        "without_rag": "Модель придумает общие правила, не связанные с компанией",  # без RAG
        "with_rag": "Модель найдёт внутренний документ компании и процитирует его"  # с RAG
    }
]

# Выводим таблицу сравнения
print("=" * 80)                                       # разделитель
print("ПРОБЛЕМА ГАЛЛЮЦИНАЦИЙ И РЕШЕНИЕ RAG")
print("=" * 80)                                       # разделитель

for i, ex in enumerate(hallucination_examples, 1):    # перебираем примеры
    print(f"\nПример {i}: {ex['question']}")          # выводим вопрос
    print(f"  БЕЗ RAG: {ex['without_rag']}")          # ответ без RAG
    print(f"  С RAG:   {ex['with_rag']}")             # ответ с RAG

print("\n" + "=" * 80)                               # разделитель
print("Вывод: RAG даёт LLM доступ к АКТУАЛЬНЫМ и ТОЧНЫМ данным!")
print("=" * 80)                                       # разделитель

In [ ]:
# ============================================================
# ШАГ 2 (продолжение): Визуализация - сравнение подходов
# ============================================================

# Создаём сравнительную диаграмму трёх подходов к обновлению знаний LLM

fig, ax = plt.subplots(figsize=(12, 6))               # создаём фигуру

# Данные для сравнения
approaches = ['RAG', 'Fine-tuning', 'Prompting']      # названия подходов
cost = [2, 9, 1]                                      # стоимость (1-10)
freshness = [9, 3, 5]                                 # свежесть данных (1-10)
accuracy = [8, 7, 4]                                  # точность (1-10)
speed = [7, 3, 9]                                     # скорость обновления (1-10)

x = np.arange(len(approaches))                        # позиции по оси X
width = 0.18                                          # ширина столбцов

# Рисуем столбцы для каждой метрики
bars1 = ax.bar(x - 1.5*width, cost, width, label='Стоимость', color='#e74c3c', alpha=0.8)      # стоимость
bars2 = ax.bar(x - 0.5*width, freshness, width, label='Свежесть данных', color='#2ecc71', alpha=0.8)  # свежесть
bars3 = ax.bar(x + 0.5*width, accuracy, width, label='Точность', color='#3498db', alpha=0.8)    # точность
bars4 = ax.bar(x + 1.5*width, speed, width, label='Скорость обновления', color='#f39c12', alpha=0.8)  # скорость

ax.set_xlabel('Подход', fontsize=13)                  # подпись оси X
ax.set_ylabel('Балл (1-10)', fontsize=13)             # подпись оси Y
ax.set_title('Сравнение подходов к обновлению знаний LLM', fontsize=15)  # заголовок
ax.set_xticks(x)                                     # позиции делений
ax.set_xticklabels(approaches, fontsize=12)           # подписи делений
ax.legend(fontsize=11)                                # легенда
ax.set_ylim(0, 11)                                   # пределы оси Y

# Добавляем значения над столбцами
for bars in [bars1, bars2, bars3, bars4]:             # перебираем группы столбцов
    for bar in bars:                                  # перебираем столбцы в группе
        height = bar.get_height()                     # высота столбца
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),  # позиция текста
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)  # параметры текста

plt.tight_layout()                                    # автоматическая подстройка отступов
plt.savefig('rag_vs_approaches.png', dpi=150, bbox_inches='tight')  # сохраняем график
plt.show()                                            # отображаем график
print("Диаграмма сравнения подходов готова!")

---
## Шаг 3. Архитектура RAG - пайплайн retrieve -> generate

RAG состоит из двух основных фаз:

### Фаза 1: Retrieval (поиск)
1. Пользователь задаёт **вопрос**
2. Вопрос преобразуется в **эмбеддинг** (вектор)
3. Вектор ищется в **векторной базе** (FAISS)
4. Возвращаются **top-k** самых релевантных чанков

### Фаза 2: Generation (генерация)
5. Найденные чанки + вопрос формируют **промпт**
6. LLM генерирует **ответ** на основе контекста
7. Ответ возвращается пользователю с **источниками**

### Полный пайплайн
```
Документы -> Чанкинг -> Эмбеддинги -> FAISS индекс
                                              |
Вопрос -> Эмбеддинг вопроса -> Поиск в FAISS -> Top-k чанков
                                                      |
Промпт = Вопрос + Top-k чанков -> LLM -> Ответ + Источники
```

In [ ]:
# ============================================================
# ШАГ 3: Визуализация архитектуры RAG-пайплайна
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))              # создаём большую фигуру
ax.set_xlim(0, 16)                                   # пределы по X
ax.set_ylim(0, 10)                                   # пределы по Y
ax.axis('off')                                       # скрываем оси
ax.set_title('Архитектура RAG: от документов к ответу', fontsize=18, fontweight='bold', pad=20)  # заголовок

# Определяем блоки пайплайна: (x, y, ширина, высота, текст, цвет)
blocks = [                                            # список блоков диаграммы
    (1, 8.5, 3, 0.8, 'Документы', '#e74c3c'),        # исходные документы
    (5, 8.5, 3, 0.8, 'Чанкинг', '#e67e22'),          # разбиение на чанки
    (9, 8.5, 3, 0.8, 'Эмбеддинги', '#f1c40f'),       # создание эмбеддингов
    (13, 8.5, 2.5, 0.8, 'FAISS', '#2ecc71'),         # векторная база

    (1, 5.5, 3, 0.8, 'Вопрос', '#9b59b6'),           # запрос пользователя
    (5, 5.5, 3, 0.8, 'Эмбеддинг\nвопроса', '#8e44ad'),  # эмбеддинг вопроса
    (9, 5.5, 3, 0.8, 'Поиск в FAISS', '#2ecc71'),    # поиск
    (13, 5.5, 2.5, 0.8, 'Top-k\nчанков', '#27ae60'),  # результаты поиска

    (5, 2.5, 3, 0.8, 'Промпт:\nВопрос+Контекст', '#3498db'),  # формирование промпта
    (9, 2.5, 3, 0.8, 'LLM', '#2980b9'),              # языковая модель
    (13, 2.5, 2.5, 0.8, 'Ответ +\nИсточники', '#1abc9c'),  # финальный ответ
]

# Рисуем блоки
for x, y, w, h, text, color in blocks:               # перебираем блоки
    rect = plt.Rectangle((x, y), w, h, linewidth=2,  # создаём прямоугольник
                         edgecolor='black', facecolor=color, alpha=0.7, zorder=2)  # параметры
    ax.add_patch(rect)                                # добавляем прямоугольник
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',  # текст в центре
            fontsize=10, fontweight='bold', zorder=3)  # параметры текста

# Рисуем стрелки между блоками
arrows = [                                            # список стрелок (откуда -> куда)
    ((4, 8.9), (5, 8.9)),                            # Документы -> Чанкинг
    ((8, 8.9), (9, 8.9)),                            # Чанкинг -> Эмбеддинги
    ((12, 8.9), (13, 8.9)),                          # Эмбеддинги -> FAISS
    ((2.5, 8.5), (2.5, 6.3)),                        # Документы (вниз)
    ((4, 5.9), (5, 5.9)),                            # Вопрос -> Эмбеддинг вопроса
    ((8, 5.9), (9, 5.9)),                            # Эмбеддинг вопроса -> Поиск
    ((12, 5.9), (13, 5.9)),                          # Поиск -> Top-k
    ((14.25, 5.5), (14.25, 3.3)),                    # Top-k (вниз)
    ((14.25, 2.9), (12, 2.9)),                       # Top-k -> Промпт
    ((8, 2.9), (9, 2.9)),                            # Промпт -> LLM
    ((12, 2.9), (13, 2.9)),                          # LLM -> Ответ
]

for start, end in arrows:                             # перебираем стрелки
    ax.annotate('', xy=end, xytext=start,             # рисуем стрелку
                arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стиль стрелки

# Подписи фаз
ax.text(0.5, 9.5, 'ФАЗА INDEXING', fontsize=14, fontweight='bold',  # подпись фазы 1
        color='#c0392b', fontstyle='italic')
ax.text(0.5, 6.5, 'ФАЗА RETRIEVAL', fontsize=14, fontweight='bold',  # подпись фазы 2
        color='#8e44ad', fontstyle='italic')
ax.text(4.5, 3.5, 'ФАЗА GENERATION', fontsize=14, fontweight='bold',  # подпись фазы 3
        color='#2980b9', fontstyle='italic')

plt.tight_layout()                                    # автоматическая подстройка отступов
plt.savefig('rag_architecture.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Схема архитектуры RAG готова!")

---
## Шаг 4. Стратегии чанкинга документов

Чанкинг (chunking) - разбиение документов на части. Это **ключевой шаг** RAG: слишком большие чанки теряют точность, слишком маленькие теряют контекст.

### Три стратегии:

| Стратегия | Принцип | Плюсы | Минусы |
|-----------|---------|-------|--------|
| **Фиксированный размер** | Режем по N символов | Простота, стабильный размер | Режет посреди предложения |
| **По предложениям** | Режем по границам предложений | Сохраняет смысл фраз | Неравномерный размер |
| **Семантический** | Группируем по смыслу | Лучшее качество | Сложнее, медленнее |

In [ ]:
# ============================================================
# ШАГ 4: Подготовка тестовых документов для чанкинга
# ============================================================

# Создаём тестовый документ для демонстрации чанкинга
test_document = (
    'Искусственный интеллект (ИИ) - это область компьютерных наук, '
    'занимающаяся созданием систем, способных выполнять задачи, требующие человеческого '
    'интеллекта. К таким задачам относятся распознавание речи, принятие решений, '
    'перевод между языками и визуальное восприятие.

'
    'Машинное обучение является подразделом ИИ, который позволяет компьютерам учиться '
    'на данных без явного программирования. Основные типы машинного обучения: '
    'обучение с учителем, обучение без учителя и обучение с подкреплением.

'
    'Глубокое обучение - это подраздел машинного обучения, использующий многослойные '
    'нейронные сети. Сверточные нейронные сети (CNN) отлично подходят для обработки '
    'изображений, а рекуррентные нейронные сети (RNN) - для последовательных данных.

'
    'Трансформеры произвели революцию в обработке естественного языка. Модель BERT '
    'использует двунаправленное внимание для понимания контекста, а GPT генерирует '
    'текст авторегрессионным способом. Эти модели обучаются на огромных корпусах текста.

'
    'RAG (Retrieval-Augmented Generation) объединяет поиск информации с генерацией '
    'текста. Вместо того чтобы полагаться только на знания, зашитые в модель, RAG '
    'извлекает релевантные документы из базы знаний и использует их как контекст '
    'для генерации более точных и актуальных ответов.'
)

print(f"Длина документа: {len(test_document)} символов")  # выводим длину
print(f"Количество предложений: {len(test_document.split('. '))}")  # количество предложений
print("\nПервые 200 символов:")
print(test_document[:200])                            # показываем начало

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Стратегия 1 - Фиксированный размер
# ============================================================

def chunk_fixed_size(text, chunk_size=200, overlap=50):
    # Разбиваем текст на чанки фиксированного размера с перекрытием
    chunks = []                                       # список чанков
    start = 0                                         # начальная позиция
    while start < len(text):                          # пока не достигли конца текста
        end = start + chunk_size                      # конечная позиция чанка
        chunk = text[start:end]                       # вырезаем чанк
        chunks.append(chunk)                          # добавляем в список
        start += chunk_size - overlap                 # сдвигаем с перекрытием
    return chunks                                     # возвращаем список чанков

# Применяем фиксированный чанкинг
fixed_chunks = chunk_fixed_size(test_document, chunk_size=200, overlap=50)  # разбиваем на чанки

print("=" * 60)                                       # разделитель
print("СТРАТЕГИЯ 1: Фиксированный размер (200 символов, overlap=50)")
print("=" * 60)                                       # разделитель

for i, chunk in enumerate(fixed_chunks):              # перебираем чанки
    print(f"\nЧанк {i+1} ({len(chunk)} символов):")  # номер и размер
    print(f"  '{chunk[:80]}...' " if len(chunk) > 80 else f"  '{chunk}'")  # содержимое

print(f"\nВсего чанков: {len(fixed_chunks)}")        # общее количество

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Стратегия 2 - По предложениям
# ============================================================

def chunk_by_sentences(text, max_chunk_size=300):
    # Разбиваем текст на чанки по границам предложений
    # Группируем предложения, пока размер чанка не превысит максимум
    sentence_endings = re.split(r'(?<=[.!?])\s+', text)  # разбиваем по концам предложений
    chunks = []                                       # список чанков
    current_chunk = ""                                # текущий формируемый чанк

    for sentence in sentence_endings:                 # перебираем предложения
        if len(current_chunk) + len(sentence) + 1 > max_chunk_size and current_chunk:  # если превышаем лимит
            chunks.append(current_chunk.strip())      # сохраняем текущий чанк
            current_chunk = sentence                  # начинаем новый
        else:
            current_chunk += " " + sentence if current_chunk else sentence  # добавляем предложение

    if current_chunk.strip():                         # если остался последний чанк
        chunks.append(current_chunk.strip())          # добавляем его

    return chunks                                     # возвращаем список чанков

# Применяем чанкинг по предложениям
sentence_chunks = chunk_by_sentences(test_document, max_chunk_size=300)  # разбиваем

print("=" * 60)                                       # разделитель
print("СТРАТЕГИЯ 2: По предложениям (max_chunk_size=300)")
print("=" * 60)                                       # разделитель

for i, chunk in enumerate(sentence_chunks):           # перебираем чанки
    print(f"\nЧанк {i+1} ({len(chunk)} символов):")  # номер и размер
    print(f"  '{chunk[:80]}...' " if len(chunk) > 80 else f"  '{chunk}'")  # содержимое

print(f"\nВсего чанков: {len(sentence_chunks)}")     # общее количество

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Стратегия 3 - Семантический чанкинг
# ============================================================

def chunk_semantic(text, model, similarity_threshold=0.5):
    # Разбиваем текст по семантической близости предложений
    # Группируем предложения с высоким сходством в один чанк
    sentence_endings = re.split(r'(?<=[.!?])\s+', text)  # разбиваем на предложения
    sentence_endings = [s.strip() for s in sentence_endings if s.strip()]  # убираем пустые

    if len(sentence_endings) <= 1:                    # если одно предложение
        return [text]                                 # возвращаем весь текст как один чанк

    # Создаём эмбеддинги для каждого предложения
    embeddings = model.encode(sentence_endings)       # получаем векторы предложений

    # Вычисляем косинусное сходство между соседними предложениями
    chunks = []                                       # список чанков
    current_sentences = [sentence_endings[0]]         # начинаем с первого предложения

    for i in range(1, len(sentence_endings)):         # перебираем остальные предложения
        # Считаем косинусное сходство текущего и предыдущего предложения
        sim = cosine_similarity(                      # вычисляем сходство
            [embeddings[i-1]], [embeddings[i]]        # пара соседних эмбеддингов
        )[0][0]                                       # извлекаем значение

        if sim < similarity_threshold:                # если сходство ниже порога
            # Сохраняем текущую группу и начинаем новую
            chunks.append(" ".join(current_sentences))  # добавляем группу
            current_sentences = [sentence_endings[i]]  # начинаем новую группу
        else:
            current_sentences.append(sentence_endings[i])  # добавляем к текущей группе

    if current_sentences:                             # если осталась последняя группа
        chunks.append(" ".join(current_sentences))    # добавляем её

    return chunks, embeddings                         # возвращаем чанки и эмбеддинги

# Загружаем маленькую модель для чанкинга (быстрая)
small_model = SentenceTransformer('all-MiniLM-L6-v2')  # легковесная модель эмбеддингов

# Применяем семантический чанкинг
semantic_chunks, sem_embeddings = chunk_semantic(     # разбиваем по смыслу
    test_document, small_model, similarity_threshold=0.3  # порог сходства
)

print("=" * 60)                                       # разделитель
print("СТРАТЕГИЯ 3: Семантический чанкинг (threshold=0.3)")
print("=" * 60)                                       # разделитель

for i, chunk in enumerate(semantic_chunks):           # перебираем чанки
    print(f"\nЧанк {i+1} ({len(chunk)} символов):")  # номер и размер
    print(f"  '{chunk[:80]}...' " if len(chunk) > 80 else f"  '{chunk}'")  # содержимое

print(f"\nВсего чанков: {len(semantic_chunks)}")     # общее количество

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Сравнение стратегий чанкинга
# ============================================================

# Визуализируем размеры чанков для каждой стратегии

fig, axes = plt.subplots(1, 3, figsize=(18, 5))      # три графика в ряд

strategies = [                                        # данные стратегий
    ("Фиксированный\n(200 символов)", fixed_chunks, '#e74c3c'),
    ("По предложениям\n(max 300)", sentence_chunks, '#3498db'),
    ("Семантический\n(threshold=0.3)", semantic_chunks, '#2ecc71'),
]

for idx, (name, chunks, color) in enumerate(strategies):  # перебираем стратегии
    ax = axes[idx]                                    # текущая ось
    sizes = [len(c) for c in chunks]                  # размеры чанков
    ax.bar(range(len(sizes)), sizes, color=color, alpha=0.7, edgecolor='black')  # столбцы
    ax.set_title(name, fontsize=13, fontweight='bold')  # заголовок
    ax.set_xlabel('Номер чанка', fontsize=11)         # подпись X
    ax.set_ylabel('Размер (символы)', fontsize=11)    # подпись Y
    ax.axhline(y=np.mean(sizes), color='black', linestyle='--',  # средний размер
               label=f'Среднее: {np.mean(sizes):.0f}')
    ax.legend(fontsize=10)                            # легенда

plt.suptitle('Сравнение стратегий чанкинга', fontsize=16, fontweight='bold', y=1.02)  # общий заголовок
plt.tight_layout()                                    # подстройка отступов
plt.savefig('chunking_comparison.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Сравнение стратегий чанкинга готово!")

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Влияние размера чанка на качество
# ============================================================

# Исследуем, как размер чанка влияет на количество чанков и их средний размер

chunk_sizes_range = range(50, 501, 25)                # диапазон размеров чанка
num_chunks_list = []                                  # количество чанков
avg_chunk_size_list = []                              # средний размер чанка

for cs in chunk_sizes_range:                          # перебираем размеры
    chunks = chunk_fixed_size(test_document, chunk_size=cs, overlap=cs//4)  # разбиваем
    num_chunks_list.append(len(chunks))               # сохраняем количество
    avg_chunk_size_list.append(np.mean([len(c) for c in chunks]))  # средний размер

fig, ax1 = plt.subplots(figsize=(12, 6))              # создаём фигуру

color1 = '#e74c3c'                                    # цвет для количества
color2 = '#3498db'                                    # цвет для среднего размера

ax1.set_xlabel('Размер чанка (символы)', fontsize=13)  # подпись X
ax1.set_ylabel('Количество чанков', color=color1, fontsize=13)  # подпись Y1
line1 = ax1.plot(chunk_sizes_range, num_chunks_list,  # график количества
                 color=color1, linewidth=2, marker='o', markersize=4, label='Количество чанков')
ax1.tick_params(axis='y', labelcolor=color1)          # цвет оси Y1

ax2 = ax1.twinx()                                     # вторая ось Y
ax2.set_ylabel('Средний размер чанка', color=color2, fontsize=13)  # подпись Y2
line2 = ax2.plot(chunk_sizes_range, avg_chunk_size_list,  # график среднего размера
                 color=color2, linewidth=2, marker='s', markersize=4, label='Средний размер')
ax2.tick_params(axis='y', labelcolor=color2)          # цвет оси Y2

lines = line1 + line2                                # объединяем линии
labels = [l.get_label() for l in lines]              # извлекаем подписи
ax1.legend(lines, labels, loc='center right', fontsize=11)  # легенда

ax1.set_title('Влияние размера чанка на разбиение документа', fontsize=15)  # заголовок
plt.tight_layout()                                    # подстройка отступов
plt.savefig('chunk_size_effect.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("График влияния размера чанка готов!")

---
## Шаг 5. Эмбеддинги документов - sentence-transformers на практике

Эмбеддинг - это **векторное представление** текста. Похожие по смыслу тексты имеют похожие векторы.

### sentence-transformers
Библиотека `sentence-transformers` предоставляет предобученные модели, которые превращают текст в плотные векторы (обычно 384 или 768 измерений).

### Ключевые концепции:
- **Косинусное сходство** = 1 для одинаковых текстов, 0 для независимых, -1 для противоположных
- **Евклидово расстояние** = 0 для одинаковых, растёт для разных
- Модель `all-MiniLM-L6-v2` - быстрая и качественная (384 измерения)

In [ ]:
# ============================================================
# ШАГ 5: Создаём эмбеддинги с помощью sentence-transformers
# ============================================================

# Используем модель all-MiniLM-L6-v2 - быструю и качественную
# Она превращает текст в векторы из 384 чисел

model = SentenceTransformer('all-MiniLM-L6-v2')      # загружаем предобученную модель
print(f"Модель загружена: all-MiniLM-L6-v2")         # подтверждаем загрузку
print(f"Размерность эмбеддинга: {model.get_sentence_embedding_dimension()}")  # размерность

# Создаём набор текстов для демонстрации эмбеддингов
sample_texts = [                                      # примеры текстов
    "Искусственный интеллект меняет мир технологий",  # про ИИ
    "Машинное обучение позволяет компьютерам учиться",  # про ML
    "Нейронные сети имитируют работу мозга",          # про нейросети
    "RAG объединяет поиск и генерацию текста",        # про RAG
    "Трансформеры произвели революцию в NLP",         # про трансформеры
    "Футбол - самый популярный спорт в мире",         # про спорт
    "Рецепт борща: свёкла, капуста, картофель",       # про еду
    "Погода сегодня солнечная и тёплая",              # про погоду
]

# Создаём эмбеддинги для всех текстов
embeddings = model.encode(sample_texts)               # кодируем тексты в векторы

print(f"\nФорма матрицы эмбеддингов: {embeddings.shape}")  # размерность матрицы
print(f"Количество текстов: {embeddings.shape[0]}")   # количество текстов
print(f"Размерность каждого вектора: {embeddings.shape[1]}")  # размерность

# Показываем первые 5 значений первого эмбеддинга
print(f"\nПервый эмбеддинг (первые 5 значений): {embeddings[0][:5]}")  # начало вектора

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Матрица сходства текстов
# ============================================================

# Вычисляем косинусное сходство между всеми парами текстов
similarity_matrix = cosine_similarity(embeddings)     # матрица сходства NxN

# Визуализируем матрицу сходства как тепловую карту
fig, ax = plt.subplots(figsize=(12, 10))              # создаём фигуру

# Сокращённые подписи для осей
short_labels = [t[:25] + "..." if len(t) > 25 else t for t in sample_texts]  # обрезаем тексты

im = ax.imshow(similarity_matrix, cmap='RdYlGn', vmin=-0.2, vmax=1)  # тепловая карта
ax.set_xticks(range(len(sample_texts)))              # деления X
ax.set_yticks(range(len(sample_texts)))              # деления Y
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)  # подписи X
ax.set_yticklabels(short_labels, fontsize=9)         # подписи Y
ax.set_title('Матрица косинусного сходства текстов', fontsize=15, fontweight='bold')  # заголовок

# Добавляем значения в ячейки
for i in range(len(sample_texts)):                   # перебираем строки
    for j in range(len(sample_texts)):               # перебираем столбцы
        val = similarity_matrix[i, j]                # значение сходства
        color = 'white' if val < 0.5 else 'black'    # цвет текста
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',  # текст
                color=color, fontsize=8)              # параметры текста

plt.colorbar(im, ax=ax, label='Косинусное сходство')  # цветовая шкала
plt.tight_layout()                                    # подстройка отступов
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Матрица сходства готова!")

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): t-SNE визуализация эмбеддингов
# ============================================================

# Снижаем размерность с 384 до 2 для визуализации
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(sample_texts)-1))  # t-SNE
embeddings_2d = tsne.fit_transform(embeddings)        # трансформируем в 2D

# Визуализируем точки в 2D пространстве
fig, ax = plt.subplots(figsize=(12, 8))               # создаём фигуру

# Цвета для разных категорий
categories = ['ИИ/ML', 'ИИ/ML', 'ИИ/ML', 'ИИ/ML', 'ИИ/ML', 'Спорт', 'Еда', 'Погода']  # категории
colors_map = {'ИИ/ML': '#3498db', 'Спорт': '#e74c3c', 'Еда': '#2ecc71', 'Погода': '#f39c12'}  # цвета
point_colors = [colors_map[cat] for cat in categories]  # цвета точек

# Рисуем точки
for i, (x, y) in enumerate(embeddings_2d):           # перебираем точки
    ax.scatter(x, y, c=point_colors[i], s=200, edgecolors='black', linewidth=1.5, zorder=3)  # точка
    ax.annotate(sample_texts[i][:20], (x, y),        # подпись точки
                textcoords="offset points", xytext=(10, 5), fontsize=9)  # параметры подписи

# Легенда для категорий
for cat, color in colors_map.items():                 # перебираем категории
    ax.scatter([], [], c=color, s=150, label=cat, edgecolors='black')  # элемент легенды
ax.legend(fontsize=11, title='Категория')             # отображаем легенду

ax.set_title('t-SNE визуализация эмбеддингов текстов', fontsize=15, fontweight='bold')  # заголовок
ax.set_xlabel('t-SNE компонента 1', fontsize=12)     # подпись X
ax.set_ylabel('t-SNE компонента 2', fontsize=12)     # подпись Y

plt.tight_layout()                                    # подстройка отступов
plt.savefig('embeddings_tsne.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("t-SNE визуализация готова!")

---
## Шаг 6. Векторная база FAISS с нуля

FAISS (Facebook AI Similarity Search) - библиотека для **быстрого поиска** ближайших векторов. Она используется в production-системах по всему миру.

### Типы индексов FAISS:
| Индекс | Описание | Когда использовать |
|--------|----------|-------------------|
| `IndexFlatL2` | Полный перебор, L2 расстояние | Малые датасеты, точный поиск |
| `IndexFlatIP` | Полный перебор, скалярное произведение | Нормализованные векторы |
| `IndexIVFFlat` | Инвертированный файл с кластеризацией | Средние датасеты |
| `IndexIVFPQ` | Инвертированный файл + квантование | Большие датасеты |

In [ ]:
# ============================================================
# ШАГ 6: Создаём FAISS индекс с нуля
# ============================================================

# Подготавливаем набор чанков для индексации
# Используем предложения из тестового документа

rag_documents = [                                     # база знаний для RAG
    "Искусственный интеллект (ИИ) - это область компьютерных наук, занимающаяся созданием интеллектуальных систем.",
    "Машинное обучение позволяет компьютерам учиться на данных без явного программирования.",
    "Глубокое обучение использует многослойные нейронные сети для решения сложных задач.",
    "Сверточные нейронные сети (CNN) отлично подходят для обработки изображений.",
    "Рекуррентные нейронные сети (RNN) предназначены для работы с последовательными данными.",
    "Трансформеры произвели революцию в обработке естественного языка в 2017 году.",
    "Модель BERT использует двунаправленное внимание для понимания контекста текста.",
    "GPT генерирует текст авторегрессионным способом, предсказывая следующий токен.",
    "RAG объединяет поиск информации с генерацией текста для более точных ответов.",
    "Векторные базы данных позволяют быстро находить похожие тексты по смыслу.",
    "FAISS - библиотека от Facebook для быстрого поиска ближайших векторов.",
    "Эмбеддинги - это числовые представления текста в многомерном пространстве.",
    "Косинусное сходство измеряет угол между двумя векторами.",
    "Чанкинг - разбиение документов на части для индексации в RAG.",
    "Prompt engineering помогает улучшить качество ответов языковых моделей.",
]

# Создаём эмбеддинги для всех документов
doc_embeddings = model.encode(rag_documents, normalize_embeddings=True)  # нормализованные эмбеддинги

print(f"Количество документов: {len(rag_documents)}")  # количество документов
print(f"Форма матрицы эмбеддингов: {doc_embeddings.shape}")  # размерность

# Создаём FAISS индекс
dimension = doc_embeddings.shape[1]                   # размерность векторов
index = faiss.IndexFlatIP(dimension)                  # индекс со скалярным произведением (inner product)

# Добавляем векторы в индекс
index.add(doc_embeddings.astype(np.float32))          # добавляем эмбеддинги

print(f"FAISS индекс создан!")                        # подтверждение
print(f"Количество векторов в индексе: {index.ntotal}")  # количество векторов

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Поиск в FAISS
# ============================================================

def search_faiss(query, model, index, documents, top_k=3):
    # Ищем top-k самых похожих документов на запрос
    query_embedding = model.encode([query], normalize_embeddings=True)  # эмбеддинг запроса
    scores, indices = index.search(query_embedding.astype(np.float32), top_k)  # поиск в FAISS
    results = []                                      # список результатов
    for score, idx in zip(scores[0], indices[0]):     # перебираем результаты
        results.append({                              # добавляем результат
            "document": documents[idx],               # текст документа
            "score": float(score),                    # оценка сходства
            "index": int(idx)                         # индекс документа
        })
    return results                                    # возвращаем результаты

# Тестируем поиск с разными запросами
test_queries = [                                      # тестовые запросы
    "Что такое нейронные сети?",                      # запрос 1
    "Как работает RAG?",                              # запрос 2
    "Для чего нужны трансформеры?",                   # запрос 3
]

for query in test_queries:                            # перебираем запросы
    results = search_faiss(query, model, index, rag_documents, top_k=3)  # ищем
    print(f"\nЗапрос: '{query}'")                    # выводим запрос
    print("-" * 60)                                   # разделитель
    for r in results:                                 # перебираем результаты
        print(f"  [score={r['score']:.4f}] {r['document'][:70]}...")  # выводим результат

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Визуализация поиска - тепловая карта запрос vs чанки
# ============================================================

# Показываем сходство между запросами и всеми чанками

query_embeddings = model.encode(test_queries, normalize_embeddings=True)  # эмбеддинги запросов

# Вычисляем матрицу сходства: запросы x документы
search_matrix = cosine_similarity(query_embeddings, doc_embeddings)  # матрица сходства

fig, ax = plt.subplots(figsize=(16, 6))               # создаём фигуру

# Сокращённые подписи документов
doc_labels = [d[:30] + "..." if len(d) > 30 else d for d in rag_documents]  # обрезаем
query_labels = [q[:30] for q in test_queries]         # подписи запросов

im = ax.imshow(search_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)  # тепловая карта
ax.set_yticks(range(len(test_queries)))               # деления Y
ax.set_yticklabels(query_labels, fontsize=11)         # подписи Y
ax.set_xticks(range(len(rag_documents)))              # деления X
ax.set_xticklabels(doc_labels, rotation=45, ha='right', fontsize=8)  # подписи X
ax.set_title('Сходство: запросы vs документы', fontsize=15, fontweight='bold')  # заголовок

# Добавляем значения
for i in range(len(test_queries)):                    # перебираем запросы
    for j in range(len(rag_documents)):               # перебираем документы
        val = search_matrix[i, j]                     # значение сходства
        color = 'white' if val > 0.6 else 'black'    # цвет текста
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',  # текст
                color=color, fontsize=7)              # параметры

plt.colorbar(im, ax=ax, label='Косинусное сходство')  # цветовая шкала
plt.tight_layout()                                    # подстройка отступов
plt.savefig('retrieval_heatmap.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Тепловая карта поиска готова!")

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Разные типы FAISS индексов
# ============================================================

# Демонстрируем разные типы индексов FAISS

# 1. IndexFlatL2 - точный поиск по L2 расстоянию
index_l2 = faiss.IndexFlatL2(dimension)               # создаём L2 индекс
index_l2.add(doc_embeddings.astype(np.float32))       # добавляем векторы

# 2. IndexFlatIP - точный поиск по скалярному произведению (косинусное сходство)
index_ip = faiss.IndexFlatIP(dimension)               # создаём IP индекс
index_ip.add(doc_embeddings.astype(np.float32))       # добавляем векторы

# 3. IndexIVFFlat - приближённый поиск с кластеризацией (для больших датасетов)
nlist = 3                                             # количество кластеров
quantizer = faiss.IndexFlatL2(dimension)              # квантователь для IVF
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist)  # IVF индекс
index_ivf.train(doc_embeddings.astype(np.float32))    # обучаем кластеризацию
index_ivf.add(doc_embeddings.astype(np.float32))      # добавляем векторы

# Сравниваем результаты всех индексов для одного запроса
test_query = "Как работает машинное обучение?"        # тестовый запрос
query_emb = model.encode([test_query], normalize_embeddings=True).astype(np.float32)  # эмбеддинг

print(f"Запрос: '{test_query}'")                     # выводим запрос
print("=" * 60)                                       # разделитель

# Поиск в каждом индексе
for name, idx_obj in [("IndexFlatL2", index_l2), ("IndexFlatIP", index_ip), ("IndexIVFFlat", index_ivf)]:  # перебираем
    scores, indices = idx_obj.search(query_emb, 3)    # ищем top-3
    print(f"\n{name}:")                              # название индекса
    for score, doc_idx in zip(scores[0], indices[0]):  # перебираем результаты
        if doc_idx >= 0:                              # если результат валидный
            print(f"  [{score:.4f}] {rag_documents[doc_idx][:60]}...")  # выводим

---
## Шаг 7. Простой RAG-пайплайн - от загрузки до генерации

Теперь соберём **полный RAG-пайплайн** из всех компонентов, которые мы изучили:

1. **Загрузка** документов
2. **Чанкинг** текста на части
3. **Эмбеддинги** каждого чанка
4. **Индексация** в FAISS
5. **Поиск** релевантных чанков по запросу
6. **Генерация** ответа на основе найденного контекста

Для генерации ответов мы будем использовать шаблон промпта, который можно подключить к любой LLM.

In [ ]:
# ============================================================
# ШАГ 7: Полный RAG-пайплайн
# ============================================================

class SimpleRAG:
    # Простой RAG-пайплайн: загрузка -> чанкинг -> эмбеддинг -> поиск -> генерация

    def __init__(self, model_name='all-MiniLM-L6-v2', chunk_size=200, overlap=50):
        self.model = SentenceTransformer(model_name)  # загружаем модель эмбеддингов
        self.chunk_size = chunk_size                  # размер чанка
        self.overlap = overlap                        # перекрытие чанков
        self.documents = []                           # исходные документы
        self.chunks = []                              # все чанки
        self.chunk_metadata = []                      # метаданные чанков (от какого документа)
        self.index = None                             # FAISS индекс
        self.dimension = self.model.get_sentence_embedding_dimension()  # размерность

    def add_documents(self, docs, source_name="default"):
        # Добавляем документы в базу знаний
        for doc in docs:                              # перебираем документы
            self.documents.append(doc)                # сохраняем оригинал
            chunks = chunk_fixed_size(doc, self.chunk_size, self.overlap)  # разбиваем
            for chunk in chunks:                      # перебираем чанки
                self.chunks.append(chunk)             # добавляем чанк
                self.chunk_metadata.append({          # сохраняем метаданные
                    "source": source_name,            # источник
                    "original_doc": doc[:100] + "..."  # начало оригинала
                })
        self._build_index()                           # перестраиваем индекс

    def _build_index(self):
        # Строим FAISS индекс из чанков
        if not self.chunks:                           # если нет чанков
            return                                    # выходим
        embeddings = self.model.encode(self.chunks, normalize_embeddings=True)  # эмбеддинги
        self.index = faiss.IndexFlatIP(self.dimension)  # создаём индекс
        self.index.add(embeddings.astype(np.float32))  # добавляем векторы

    def search(self, query, top_k=3):
        # Ищем релевантные чанки
        query_emb = self.model.encode([query], normalize_embeddings=True)  # эмбеддинг запроса
        scores, indices = self.index.search(query_emb.astype(np.float32), top_k)  # поиск
        results = []                                  # список результатов
        for score, idx in zip(scores[0], indices[0]):  # перебираем результаты
            if idx >= 0 and idx < len(self.chunks):   # если индекс валидный
                results.append({                      # добавляем результат
                    "chunk": self.chunks[idx],         # текст чанка
                    "score": float(score),             # оценка
                    "metadata": self.chunk_metadata[idx]  # метаданные
                })
        return results                                # возвращаем результаты

    def generate_answer(self, query, top_k=3):
        # Генерируем ответ на основе найденного контекста
        results = self.search(query, top_k)           # ищем релевантные чанки

        # Формируем контекст из найденных чанков
        context = "\n\n".join([                      # объединяем чанки
            f"[Источник {i+1}]: {r['chunk']}"          # с номерами источников
            for i, r in enumerate(results)
        ])

        # Формируем промпт для LLM
        prompt = (                                      # шаблон промпта
            'Ответь на вопрос на основе предоставленного контекста.
'
            'Если ответ нельзя найти в контексте, скажи, что информации недостаточно.

'
            'Контекст:
' + context + '

'
            'Вопрос: ' + query + '

Ответ:'
        )

        return {                                      # возвращаем результат
            "query": query,                           # исходный запрос
            "answer_prompt": prompt,                  # промпт для LLM
            "sources": results,                       # источники
            "context": context                        # собранный контекст
        }

    def get_stats(self):
        # Возвращаем статистику RAG-системы
        return {                                      # словарь статистики
            "documents": len(self.documents),          # количество документов
            "chunks": len(self.chunks),                # количество чанков
            "index_size": self.index.ntotal if self.index else 0,  # размер индекса
            "dimension": self.dimension,              # размерность
            "chunk_size": self.chunk_size,            # размер чанка
            "overlap": self.overlap                   # перекрытие
        }

# Создаём RAG-систему
rag = SimpleRAG(chunk_size=200, overlap=50)           # инициализируем RAG
print("RAG-пайплайн создан!")

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Загружаем документы в RAG
# ============================================================

# Добавляем документы из разных источников

# Источник 1: Статья про ИИ
ai_docs = [                                           # документы про ИИ
    "Искусственный интеллект - это область компьютерных наук, занимающаяся созданием систем, способных выполнять задачи, требующие человеческого интеллекта. Включает распознавание речи, принятие решений и перевод.",
    "Машинное обучение - подраздел ИИ, позволяющий системам учиться на данных. Основные типы: обучение с учителем (классификация, регрессия), без учителя (кластеризация) и с подкреплением.",
    "Глубокое обучение использует многослойные нейронные сети. Сверточные сети (CNN) обрабатывают изображения, рекуррентные (RNN) - последовательности, трансформеры - текст.",
]

# Источник 2: Статья про трансформеры
transformer_docs = [                                  # документы про трансформеры
    "Трансформеры - архитектура нейросетей, основанная на механизме внимания. Представлены в 2017 году в статье Attention Is All You Need. Стали стандартом в NLP.",
    "BERT - двунаправленная модель на основе трансформера. Понимает контекст с обеих сторон. Используется для классификации, NER, ответов на вопросы.",
    "GPT - авторегрессионная модель на основе трансформера. Генерирует текст, предсказывая следующий токен. GPT-3 имеет 175 миллиардов параметров.",
]

# Источник 3: Статья про RAG
rag_docs = [                                          # документы про RAG
    "RAG (Retrieval-Augmented Generation) - подход, объединяющий поиск информации с генерацией текста. Позволяет LLM использовать внешние знания.",
    "Векторные базы данных хранят эмбеддинги документов и обеспечивают быстрый поиск. Популярные решения: FAISS, Pinecone, Weaviate, ChromaDB.",
    "Чанкинг - разбиение документов на части для индексации. Стратегии: фиксированный размер, по предложениям, семантический. Влияет на качество RAG.",
]

# Добавляем все документы
rag.add_documents(ai_docs, source_name="Статья_ИИ")   # добавляем документы про ИИ
rag.add_documents(transformer_docs, source_name="Статья_Трансформеры")  # про трансформеры
rag.add_documents(rag_docs, source_name="Статья_RAG")  # про RAG

# Выводим статистику
stats = rag.get_stats()                               # получаем статистику
print("Статистика RAG-системы:")
for key, value in stats.items():                      # перебираем статистику
    print(f"  {key}: {value}")                        # выводим

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Тестируем RAG-пайплайн
# ============================================================

# Задаём вопросы и смотрим результаты поиска

test_questions = [                                    # тестовые вопросы
    "Что такое машинное обучение?",                   # вопрос 1
    "Как работает RAG?",                              # вопрос 2
    "Чем BERT отличается от GPT?",                    # вопрос 3
    "Какие бывают стратегии чанкинга?",               # вопрос 4
]

for question in test_questions:                       # перебираем вопросы
    result = rag.generate_answer(question, top_k=3)   # генерируем ответ
    print(f"\n{'='*60}")                             # разделитель
    print(f"ВОПРОС: {question}")                      # выводим вопрос
    print(f"{'='*60}")                                # разделитель
    print(f"\nНайденные источники:")                 # заголовок источников
    for i, source in enumerate(result['sources'], 1):  # перебираем источники
        print(f"  [{i}] (score={source['score']:.4f}) Источник: {source['metadata']['source']}")  # источник
        print(f"      {source['chunk'][:80]}...")     # текст чанка
    print(f"\nСформированный промпт (первые 300 символов):")  # заголовок промпта
    print(f"  {result['answer_prompt'][:300]}...")    # промпт

---
## Шаг 8. FastAPI сервер для RAG-запросов

Теперь мы создадим **полноценный сервер**, который:
- Загружает документы и создаёт FAISS индекс при старте
- `/query` - принимает вопрос, возвращает ответ с источниками
- `/add_document` - добавляет новый документ в базу знаний
- `/search` - возвращает top-k релевантных чанков без генерации

После запуска вы сможете отправлять запросы через curl!

In [ ]:
# ============================================================
# ШАГ 8: Создаём FastAPI приложение для RAG
# ============================================================

# Записываем код сервера в файл
server_code = '''
import uvicorn                                         # ASGI-сервер
from fastapi import FastAPI                            # фреймворк для API
from pydantic import BaseModel                         # валидация данных
from sentence_transformers import SentenceTransformer  # модель эмбеддингов
import faiss                                           # векторный поиск
import numpy as np                                     # массивы и математика
import re                                              # регулярные выражения
from typing import List, Optional                      # типы для аннотаций

# Создаём FastAPI приложение
app = FastAPI(title="RAG Server", version="1.0")      # приложение с заголовком

# Глобальные переменные для RAG
model = SentenceTransformer("all-MiniLM-L6-v2")       # модель эмбеддингов
documents = []                                         # список документов
chunks = []                                            # список чанков
chunk_sources = []                                     # источники чанков
dimension = model.get_sentence_embedding_dimension()  # размерность
index = faiss.IndexFlatIP(dimension)                   # FAISS индекс

# Модели данных для API
class QueryRequest(BaseModel):                         # запрос с вопросом
    question: str                                      # текст вопроса
    top_k: int = 3                                     # количество результатов
    temperature: float = 0.7                           # температура генерации

class DocumentRequest(BaseModel):                      # запрос на добавление документа
    text: str                                          # текст документа
    source: str = "manual"                             # источник

class SearchRequest(BaseModel):                        # запрос на поиск
    query: str                                         # поисковый запрос
    top_k: int = 5                                     # количество результатов

def chunk_text(text, chunk_size=200, overlap=50):      # функция чанкинга
    chunks_list = []                                   # список чанков
    start = 0                                          # начальная позиция
    while start < len(text):                           # пока не конец
        end = start + chunk_size                       # конечная позиция
        chunks_list.append(text[start:end])            # добавляем чанк
        start += chunk_size - overlap                  # сдвигаем с перекрытием
    return chunks_list                                 # возвращаем список

def add_to_index(text, source="manual"):               # добавление в индекс
    global index                                       # глобальный индекс
    documents.append(text)                             # сохраняем документ
    new_chunks = chunk_text(text)                      # разбиваем на чанки
    for c in new_chunks:                               # перебираем чанки
        chunks.append(c)                               # добавляем в список
        chunk_sources.append(source)                   # сохраняем источник
    # Перестраиваем индекс
    all_embeddings = model.encode(chunks, normalize_embeddings=True)  # эмбеддинги
    index = faiss.IndexFlatIP(dimension)               # новый индекс
    index.add(all_embeddings.astype(np.float32))       # добавляем все векторы

@app.on_event("startup")                              # при запуске сервера
async def startup():                                   # функция инициализации
    # Добавляем начальные документы
    initial_docs = [
        ("Искусственный интеллект - область компьютерных наук о создании интеллектуальных систем.", "wiki"),
        ("Машинное обучение позволяет компьютерам учиться на данных без явного программирования.", "wiki"),
        ("Глубокое обучение использует многослойные нейронные сети для решения задач.", "wiki"),
        ("RAG объединяет поиск информации с генерацией текста для точных ответов.", "paper"),
        ("FAISS - библиотека для быстрого поиска ближайших векторов от Facebook.", "docs"),
        ("Трансформеры произвели революцию в обработке естественного языка в 2017.", "paper"),
        ("BERT - двунаправленная модель для понимания контекста текста.", "paper"),
        ("GPT - авторегрессионная модель для генерации текста.", "paper"),
    ]
    for text, src in initial_docs:                     # перебираем документы
        add_to_index(text, src)                        # добавляем каждый
    print(f"RAG Server готов! Индекс содержит {index.ntotal} чанков")

@app.get("/")                                         # корневой эндпоинт
async def root():                                     # приветствие
    return {"message": "RAG Server работает! Используйте /docs для API"}

@app.post("/query")                                   # эндпоинт запроса
async def query_rag(request: QueryRequest):            # обработка вопроса
    query_emb = model.encode([request.question], normalize_embeddings=True)  # эмбеддинг вопроса
    scores, indices = index.search(query_emb.astype(np.float32), request.top_k)  # поиск
    sources = []                                      # список источников
    for score, idx in zip(scores[0], indices[0]):      # перебираем результаты
        if idx >= 0 and idx < len(chunks):             # если индекс валидный
            sources.append({                           # добавляем источник
                "chunk": chunks[idx],                  # текст чанка
                "score": float(score),                 # оценка
                "source": chunk_sources[idx]           # источник
            })
    context = "\n".join([s["chunk"] for s in sources])  # собираем контекст
    return {                                          # возвращаем результат
        "question": request.question,                  # исходный вопрос
        "answer_prompt": f"Контекст:\n{context}\n\nВопрос: {request.question}\nОтвет:",  # промпт
        "sources": sources,                            # источники
        "num_sources": len(sources)                    # количество источников
    }

@app.post("/add_document")                            # эндпоинт добавления документа
async def add_document(request: DocumentRequest):      # обработка добавления
    add_to_index(request.text, request.source)         # добавляем в индекс
    return {                                          # возвращаем подтверждение
        "message": "Документ добавлен!",               # сообщение
        "total_chunks": len(chunks),                   # всего чанков
        "total_documents": len(documents)              # всего документов
    }

@app.post("/search")                                  # эндпоинт поиска
async def search_rag(request: SearchRequest):          # обработка поиска
    query_emb = model.encode([request.query], normalize_embeddings=True)  # эмбеддинг
    scores, indices = index.search(query_emb.astype(np.float32), request.top_k)  # поиск
    results = []                                      # список результатов
    for score, idx in zip(scores[0], indices[0]):      # перебираем результаты
        if idx >= 0 and idx < len(chunks):             # если индекс валидный
            results.append({                           # добавляем результат
                "chunk": chunks[idx],                  # текст чанка
                "score": float(score),                 # оценка
                "source": chunk_sources[idx],           # источник
                "index": int(idx)                       # индекс
            })
    return {"query": request.query, "results": results}  # возвращаем

@app.get("/stats")                                    # эндпоинт статистики
async def get_stats():                                 # обработка статистики
    return {                                          # возвращаем статистику
        "documents": len(documents),                   # количество документов
        "chunks": len(chunks),                         # количество чанков
        "index_size": index.ntotal,                    # размер индекса
        "dimension": dimension                         # размерность
    }
'''

# Записываем сервер в файл
with open('rag_server.py', 'w', encoding='utf-8') as f:  # открываем файл для записи
    f.write(server_code)                              # записываем код сервера

print("Файл rag_server.py создан!")                   # подтверждение

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Запускаем RAG-сервер с ngrok
# ============================================================

import subprocess                                      # для запуска процессов
import time                                           # для задержек
import requests                                       # для HTTP-запросов

# Запускаем сервер в фоновом процессе
server_process = subprocess.Popen(                    # запускаем процесс
    ["python", "rag_server.py"],                      # команда запуска
    stdout=subprocess.PIPE,                           # перехватываем stdout
    stderr=subprocess.PIPE                            # перехватываем stderr
)

time.sleep(5)                                         # ждём запуска сервера

# Запускаем ngrok для публичного доступа
from pyngrok import ngrok                            # импортируем ngrok

# Раскомментируйте и добавьте ваш токен ngrok:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")           # токен ngrok

try:                                                  # пытаемся запустить ngrok
    public_url = ngrok.connect(8000)                  # открываем туннель на порт 8000
    print(f"Сервер RAG запущен!")                     # подтверждение
    print(f"Публичный URL: {public_url}")             # публичный адрес
    print(f"Локальный URL: http://localhost:8000")    # локальный адрес
except Exception as e:                                # если ошибка ngrok
    print(f"ngrok не запущен: {e}")                   # сообщение об ошибке
    print("Сервер работает локально на http://localhost:8000")  # локальный адрес

# Проверяем, что сервер работает
time.sleep(3)                                         # ждём инициализации
try:
    response = requests.get("http://localhost:8000/")  # проверяем корневой эндпоинт
    print(f"\nПроверка сервера: {response.json()}")  # выводим ответ
except Exception as e:                                # если ошибка
    print(f"Сервер ещё запускается: {e}")             # сообщение

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Тестируем RAG-сервер через HTTP
# ============================================================

# Тестируем все эндпоинты сервера

import requests                                       # для HTTP-запросов
import json                                           # для форматирования JSON

base_url = "http://localhost:8000"                    # базовый URL сервера

# 1. Тестируем /query - задаём вопрос
print("=" * 60)                                       # разделитель
print("1. ЭНДПОИНТ /query - Задаём вопрос")
print("=" * 60)                                       # разделитель

query_data = {                                        # данные запроса
    "question": "Что такое машинное обучение?",        # вопрос
    "top_k": 3,                                       # количество результатов
    "temperature": 0.7                                # температура
}

try:
    response = requests.post(f"{base_url}/query", json=query_data)  # отправляем запрос
    result = response.json()                          # парсим JSON
    print(f"Вопрос: {result['question']}")            # выводим вопрос
    print(f"Количество источников: {result['num_sources']}")  # количество источников
    for i, src in enumerate(result['sources'], 1):    # перебираем источники
        print(f"  [{i}] (score={src['score']:.4f}) {src['chunk'][:60]}...")  # источник
except Exception as e:                                # если ошибка
    print(f"Ошибка: {e}")                             # сообщение

# 2. Тестируем /add_document - добавляем документ
print(f"\n{'='*60}")                                 # разделитель
print("2. ЭНДПОИНТ /add_document - Добавляем документ")
print("=" * 60)                                       # разделитель

new_doc = {                                           # новый документ
    "text": "LangChain - фреймворк для создания приложений с LLM. Предоставляет инструменты для чанкинга, эмбеддингов и оркестрации RAG-пайплайнов.",  # текст
    "source": "manual_add"                            # источник
}

try:
    response = requests.post(f"{base_url}/add_document", json=new_doc)  # добавляем
    result = response.json()                          # парсим JSON
    print(f"Сообщение: {result['message']}")          # подтверждение
    print(f"Всего чанков: {result['total_chunks']}")  # статистика
    print(f"Всего документов: {result['total_documents']}")  # статистика
except Exception as e:                                # если ошибка
    print(f"Ошибка: {e}")                             # сообщение

# 3. Тестируем /search - поиск без генерации
print(f"\n{'='*60}")                                 # разделитель
print("3. ЭНДПОИНТ /search - Поиск чанков")
print("=" * 60)                                       # разделитель

search_data = {                                       # данные поиска
    "query": "фреймворк для LLM",                     # поисковый запрос
    "top_k": 3                                        # количество результатов
}

try:
    response = requests.post(f"{base_url}/search", json=search_data)  # ищем
    result = response.json()                          # парсим JSON
    print(f"Запрос: {result['query']}")               # выводим запрос
    for i, r in enumerate(result['results'], 1):      # перебираем результаты
        print(f"  [{i}] (score={r['score']:.4f}) {r['chunk'][:60]}...")  # результат
except Exception as e:                                # если ошибка
    print(f"Ошибка: {e}")                             # сообщение

# 4. Тестируем /stats - статистика
print(f"\n{'='*60}")                                 # разделитель
print("4. ЭНДПОИНТ /stats - Статистика")
print("=" * 60)                                       # разделитель

try:
    response = requests.get(f"{base_url}/stats")      # запрашиваем статистику
    result = response.json()                          # парсим JSON
    for key, value in result.items():                 # перебираем статистику
        print(f"  {key}: {value}")                    # выводим
except Exception as e:                                # если ошибка
    print(f"Ошибка: {e}")                             # сообщение

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Команды curl для самостоятельного тестирования
# ============================================================

# Скопируйте и выполните эти команды в новой ячейке терминала

# Команда 1: Задать вопрос
curl_command_1 = '''
curl -X POST "http://localhost:8000/query" \
  -H "Content-Type: application/json" \
  -d '{"question": "Что такое FAISS?", "top_k": 3, "temperature": 0.7}'
'''

# Команда 2: Добавить новый документ
curl_command_2 = '''
curl -X POST "http://localhost:8000/add_document" \
  -H "Content-Type: application/json" \
  -d '{"text": "ChromaDB - это открытая векторная база данных для AI-приложений.", "source": "user_input"}'
'''

# Команда 3: Поиск без генерации
curl_command_3 = '''
curl -X POST "http://localhost:8000/search" \
  -H "Content-Type: application/json" \
  -d '{"query": "векторная база данных", "top_k": 5}'
'''

# Команда 4: Статистика
curl_command_4 = '''
curl -X GET "http://localhost:8000/stats"
'''

print("Команды curl для тестирования сервера:")
print("=" * 60)                                       # разделитель
print("\n1. Задать вопрос:")
print(curl_command_1)                                 # команда 1
print("\n2. Добавить документ:")
print(curl_command_2)                                 # команда 2
print("\n3. Поиск чанков:")
print(curl_command_3)                                 # команда 3
print("\n4. Статистика:")
print(curl_command_4)                                 # команда 4
print("\nСкопируйте нужную команду и выполните в терминале!")  # подсказка

---
## Шаг 9. Интерактивный RAG-эксплорер

Исследуйте RAG-пайплайн в интерактивном режиме! Меняйте параметры и сразу видите результаты:

**Виджеты:**
- `chunk_size` (100-1000) - размер чанка в символах
- `top_k` (1-10) - количество возвращаемых чанков
- `temperature` (0.0-2.0) - температура генерации (для LLM)
- `similarity_threshold` (0.0-1.0) - минимальный порог сходства

In [ ]:
# ============================================================
# ШАГ 9: Интерактивный RAG-эксплорер с виджетами
# ============================================================

# Создаём расширенную базу документов для интерактивного исследования
extended_documents = [                                # расширенная база знаний
    "Искусственный интеллект - область компьютерных наук о создании интеллектуальных систем. Включает машинное обучение, обработку языка, компьютерное зрение и робототехнику.",
    "Машинное обучение позволяет компьютерам учиться на данных. Обучение с учителем использует размеченные данные, без учителя - неразмеченные, с подкреплением - награды.",
    "Глубокое обучение - подраздел машинного обучения с многослойными нейронными сетями. Сверточные сети обрабатывают изображения, рекуррентные - последовательности, трансформеры - текст.",
    "Трансформеры - архитектура 2017 года на основе механизма внимания. Позволили параллелизировать обучение и работать с длинными последовательностями.",
    "BERT - двунаправленная модель от Google. Понимает контекст слова с обеих сторон. Используется для классификации текстов, NER и ответов на вопросы.",
    "GPT - авторегрессионная модель от OpenAI. Генерирует текст токен за токеном. GPT-4 может обрабатывать текст и изображения.",
    "RAG - подход, объединяющий поиск и генерацию. Извлекает релевантные документы из базы знаний и передаёт их как контекст языковой модели.",
    "FAISS - библиотека Facebook для векторного поиска. Поддерживает точный и приближённый поиск, работает на CPU и GPU.",
    "Векторные базы данных хранят эмбеддинги и обеспечивают быстрый поиск. Популярные: FAISS, Pinecone, Weaviate, ChromaDB, Milvus.",
    "Чанкинг - разбиение документов на части. Фиксированный размер прост, по предложениям сохраняет смысл, семантический группирует по смыслу.",
    "Эмбеддинги - числовые представления текста. Похожие тексты имеют похожие векторы. Модели: all-MiniLM-L6-v2, paraphrase-multilingual-MiniLM-L12-v2.",
    "Косинусное сходство измеряет угол между векторами. Значение 1 = одинаковые, 0 = независимые, -1 = противоположные.",
    "Prompt engineering - искусство составления промптов для LLM. Включает few-shot, chain-of-thought, системные промпты.",
    "Fine-tuning - дообучение модели на специфичных данных. Требует вычислительных ресурсов, но адаптирует модель под задачу.",
    "RLHF (Reinforcement Learning from Human Feedback) - обучение с подкреплением на основе человеческих оценок. Используется для выравнивания ChatGPT.",
]

# Создаём виджеты
chunk_size_slider = widgets.IntSlider(                # слайдер размера чанка
    value=200,                                        # начальное значение
    min=100,                                          # минимум
    max=1000,                                         # максимум
    step=50,                                          # шаг
    description='chunk_size:',                        # подпись
    style={'description_width': 'initial'},           # стиль
    layout=Layout(width='400px')                      # размер
)

top_k_slider = widgets.IntSlider(                     # слайдер top_k
    value=3,                                          # начальное значение
    min=1,                                            # минимум
    max=10,                                           # максимум
    step=1,                                           # шаг
    description='top_k:',                             # подпись
    style={'description_width': 'initial'},           # стиль
    layout=Layout(width='400px')                      # размер
)

temperature_slider = widgets.FloatSlider(             # слайдер температуры
    value=0.7,                                        # начальное значение
    min=0.0,                                          # минимум
    max=2.0,                                          # максимум
    step=0.1,                                         # шаг
    description='temperature:',                       # подпись
    style={'description_width': 'initial'},           # стиль
    layout=Layout(width='400px')                      # размер
)

similarity_slider = widgets.FloatSlider(              # слайдер порога сходства
    value=0.0,                                        # начальное значение
    min=0.0,                                          # минимум
    max=1.0,                                          # максимум
    step=0.05,                                        # шаг
    description='sim_threshold:',                     # подпись
    style={'description_width': 'initial'},           # стиль
    layout=Layout(width='400px')                      # размер
)

query_text = widgets.Text(                            # текстовое поле для запроса
    value='Что такое RAG?',                           # начальный запрос
    description='Запрос:',                            # подпись
    style={'description_width': 'initial'},           # стиль
    layout=Layout(width='500px')                      # размер
)

output_area = widgets.Output()                        # область вывода результатов

def explore_rag(chunk_size, top_k, temperature, similarity_threshold, query):
    # Функция для интерактивного исследования RAG
    with output_area:                                 # выводим в область
        clear_output(wait=True)                       # очищаем предыдущий вывод

        # Создаём чанки с текущими параметрами
        all_chunks = []                               # все чанки
        for doc in extended_documents:                # перебираем документы
            chunks = chunk_fixed_size(doc, chunk_size, chunk_size // 4)  # разбиваем
            all_chunks.extend(chunks)                 # добавляем чанки

        # Создаём эмбеддинги
        chunk_embeddings = model.encode(all_chunks, normalize_embeddings=True)  # эмбеддинги

        # Создаём FAISS индекс
        faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])  # индекс
        faiss_index.add(chunk_embeddings.astype(np.float32))  # добавляем

        # Ищем релевантные чанки
        query_emb = model.encode([query], normalize_embeddings=True)  # эмбеддинг запроса
        scores, indices = faiss_index.search(query_emb.astype(np.float32), top_k)  # поиск

        # Фильтруем по порогу сходства
        print(f"RAG-эксплорер | chunk_size={chunk_size} | top_k={top_k} | temp={temperature} | threshold={similarity_threshold}")
        print("=" * 70)                               # разделитель
        print(f"Запрос: '{query}'")                   # выводим запрос
        print(f"Всего чанков: {len(all_chunks)}")     # количество чанков

        filtered_count = 0                            # счётчик прошедших фильтр
        for score, idx in zip(scores[0], indices[0]):  # перебираем результаты
            if idx >= 0 and idx < len(all_chunks):    # если индекс валидный
                if score >= similarity_threshold:      # если выше порога
                    filtered_count += 1                # увеличиваем счётчик
                    print(f"\n  Результат {filtered_count}: score={score:.4f}")
                    print(f"  {all_chunks[idx][:100]}...")
                else:                                 # если ниже порога
                    print(f"\n  [ОТФИЛЬТРОВАНО] score={score:.4f} < threshold={similarity_threshold}")

        if filtered_count == 0:                        # если нет результатов
            print("\n  Нет результатов выше порога! Попробуйте уменьшить similarity_threshold.")

        print(f"\nПрошло фильтр: {filtered_count} из {top_k} результатов")  # итоги

# Создаём интерактивный интерфейс
def on_change(change):                                # обработчик изменений
    explore_rag(                                      # обновляем результаты
        chunk_size_slider.value,                      # размер чанка
        top_k_slider.value,                           # top_k
        temperature_slider.value,                     # температура
        similarity_slider.value,                      # порог сходства
        query_text.value                              # запрос
    )

# Подключаем обработчики
chunk_size_slider.observe(on_change, names='value')   # слайдер размера
top_k_slider.observe(on_change, names='value')        # слайдер top_k
temperature_slider.observe(on_change, names='value')  # слайдер температуры
similarity_slider.observe(on_change, names='value')   # слайдер порога
query_text.observe(on_change, names='value')          # текстовое поле

# Отображаем виджеты
controls = widgets.VBox([                             # вертикальная группа виджетов
    query_text,                                       # поле запроса
    chunk_size_slider,                                # размер чанка
    top_k_slider,                                     # top_k
    temperature_slider,                               # температура
    similarity_slider,                                # порог сходства
])

display(widgets.VBox([controls, output_area]))        # отображаем всё

# Начальный запуск
explore_rag(200, 3, 0.7, 0.0, "Что такое RAG?")     # начальные параметры

---
## Шаг 10. Оценка качества RAG - precision@k, recall, MRR

Как понять, хорошо ли работает ваш RAG? Нужны **метрики оценки**:

### Метрики поиска (Retrieval):
| Метрика | Формула | Что измеряет |
|---------|---------|-------------|
| **Precision@k** | релевантные / k | Доля релевантных среди top-k |
| **Recall@k** | релевантные_top_k / все_релевантные | Доля найденных из всех релевантных |
| **MRR** | 1 / rank_первого_релевантного | Средний обратный ранг |
| **Hit Rate** | есть_релевантный / всего_запросов | Доля запросов с хотя бы одним попаданием |

In [ ]:
# ============================================================
# ШАГ 10: Метрики оценки качества поиска
# ============================================================

def precision_at_k(retrieved_relevant, k):
    # Precision@k: доля релевантных среди top-k результатов
    # retrieved_relevant: список True/False для каждого результата
    top_k = retrieved_relevant[:k]                    # берём top-k
    if len(top_k) == 0:                               # если пусто
        return 0.0                                    # возвращаем 0
    return sum(top_k) / len(top_k)                    # доля релевантных

def recall_at_k(retrieved_relevant, total_relevant, k):
    # Recall@k: доля найденных релевантных из всех релевантных
    top_k = retrieved_relevant[:k]                    # берём top-k
    if total_relevant == 0:                           # если нет релевантных
        return 0.0                                    # возвращаем 0
    return sum(top_k) / total_relevant                # доля найденных

def mean_reciprocal_rank(all_retrieved_relevant):
    # MRR: средний обратный ранг первого релевантного результата
    rr_sum = 0.0                                      # сумма обратных рангов
    for retrieved in all_retrieved_relevant:           # перебираем запросы
        for i, is_rel in enumerate(retrieved, 1):     # перебираем результаты
            if is_rel:                                # если релевантный
                rr_sum += 1.0 / i                     # добавляем 1/rank
                break                                 # берём только первый
    return rr_sum / len(all_retrieved_relevant) if all_retrieved_relevant else 0.0  # среднее

def hit_rate(all_retrieved_relevant):
    # Hit Rate: доля запросов, где есть хотя бы один релевантный
    hits = sum(1 for retrieved in all_retrieved_relevant if any(retrieved))  # считаем попадания
    return hits / len(all_retrieved_relevant) if all_retrieved_relevant else 0.0  # доля

# Демонстрируем на примере
print("=" * 60)                                       # разделитель
print("ДЕМОНСТРАЦИЯ МЕТРИК ОЦЕНКИ ПОИСКА")
print("=" * 60)                                       # разделитель

# Пример: 5 запросов, для каждого указано, релевантен ли результат
# True = релевантный, False = нерелевантный
example_relevance = [                                 # релевантность для каждого запроса
    [True, True, False, False, True],                # запрос 1: 3 релевантных из 5
    [False, True, True, False, False],               # запрос 2: 2 релевантных
    [True, False, False, False, False],              # запрос 3: 1 релевантный
    [False, False, True, True, True],                # запрос 4: 3 релевантных
    [True, True, True, False, False],                # запрос 5: 3 релевантных
]

total_relevant_per_query = [3, 4, 2, 3, 5]           # всего релевантных документов

k = 3                                                 # считаем метрики для top-3

for i, (rel, total) in enumerate(zip(example_relevance, total_relevant_per_query)):  # перебираем
    p = precision_at_k(rel, k)                        # precision@3
    r = recall_at_k(rel, total, k)                    # recall@3
    print(f"Запрос {i+1}: P@{k}={p:.2f}, R@{k}={r:.2f} | релевантность: {rel[:k]}")  # выводим

mrr = mean_reciprocal_rank(example_relevance)         # MRR
hr = hit_rate(example_relevance)                      # Hit Rate
avg_p = np.mean([precision_at_k(r, k) for r in example_relevance])  # средний precision
avg_r = np.mean([recall_at_k(r, t, k) for r, t in zip(example_relevance, total_relevant_per_query)])  # средний recall

print(f"\nСредние метрики (k={k}):")                 # заголовок
print(f"  Avg Precision@{k}: {avg_p:.3f}")            # средний precision
print(f"  Avg Recall@{k}:    {avg_r:.3f}")            # средний recall
print(f"  MRR:               {mrr:.3f}")              # MRR
print(f"  Hit Rate:          {hr:.3f}")                # Hit Rate

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Оценка нашего RAG-пайплайна
# ============================================================

# Создаём тестовый набор для оценки RAG
# Для каждого вопроса указываем, какие документы релевантны

test_set = [                                          # тестовый набор
    {
        "query": "Что такое машинное обучение?",       # вопрос
        "relevant_indices": [0, 1],                   # индексы релевантных документов
    },
    {
        "query": "Какие бывают нейронные сети?",       # вопрос
        "relevant_indices": [2],                      # индексы релевантных
    },
    {
        "query": "Что такое RAG и как он работает?",   # вопрос
        "relevant_indices": [6, 9],                   # индексы релевантных
    },
    {
        "query": "Для чего нужен FAISS?",              # вопрос
        "relevant_indices": [7, 8],                   # индексы релевантных
    },
    {
        "query": "Чем BERT отличается от GPT?",        # вопрос
        "relevant_indices": [4, 5],                   # индексы релевантных
    },
]

# Оцениваем RAG для разных значений top_k
k_values = [1, 2, 3, 5]                              # значения k для оценки
results_by_k = {}                                     # результаты по k

for k in k_values:                                    # перебираем k
    precisions = []                                   # precision для каждого запроса
    recalls = []                                      # recall для каждого запроса

    for test_item in test_set:                        # перебираем тестовые вопросы
        query = test_item["query"]                    # вопрос
        relevant = test_item["relevant_indices"]      # релевантные индексы

        # Ищем через RAG
        search_results = search_faiss(query, model, index, rag_documents, top_k=k)  # ищем

        # Определяем, какие результаты релевантны
        retrieved_relevant = []                       # релевантность результатов
        for r in search_results:                      # перебираем результаты
            is_relevant = r['index'] in relevant      # релевантен ли?
            retrieved_relevant.append(is_relevant)    # добавляем

        p = precision_at_k(retrieved_relevant, k)     # precision
        r_val = recall_at_k(retrieved_relevant, len(relevant), k)  # recall
        precisions.append(p)                          # сохраняем
        recalls.append(r_val)                         # сохраняем

    results_by_k[k] = {                               # сохраняем результаты для k
        "precision": np.mean(precisions),             # средний precision
        "recall": np.mean(recalls),                   # средний recall
    }

# Выводим результаты
print("Оценка RAG-пайплайна для разных top_k:")
print("=" * 50)                                       # разделитель
for k, metrics in results_by_k.items():               # перебираем k
    print(f"  k={k}: Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}")  # выводим

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Визуализация метрик оценки
# ============================================================

# Строим графики метрик для разных k

fig, axes = plt.subplots(1, 2, figsize=(14, 6))      # два графика

k_list = list(results_by_k.keys())                    # список k
precisions = [results_by_k[k]['precision'] for k in k_list]  # precision
recalls = [results_by_k[k]['recall'] for k in k_list]  # recall

# График 1: Precision@k
ax1 = axes[0]                                        # первая ось
bars1 = ax1.bar([str(k) for k in k_list], precisions, color='#3498db', alpha=0.8, edgecolor='black')  # столбцы
ax1.set_xlabel('k (top-k)', fontsize=13)              # подпись X
ax1.set_ylabel('Precision@k', fontsize=13)            # подпись Y
ax1.set_title('Precision@k: доля релевантных среди top-k', fontsize=14, fontweight='bold')  # заголовок
ax1.set_ylim(0, 1.1)                                 # пределы Y
for bar, val in zip(bars1, precisions):               # перебираем столбцы
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,  # позиция текста
             f'{val:.2f}', ha='center', fontsize=11)  # значение

# График 2: Recall@k
ax2 = axes[1]                                        # вторая ось
bars2 = ax2.bar([str(k) for k in k_list], recalls, color='#2ecc71', alpha=0.8, edgecolor='black')  # столбцы
ax2.set_xlabel('k (top-k)', fontsize=13)              # подпись X
ax2.set_ylabel('Recall@k', fontsize=13)               # подпись Y
ax2.set_title('Recall@k: доля найденных из всех релевантных', fontsize=14, fontweight='bold')  # заголовок
ax2.set_ylim(0, 1.1)                                 # пределы Y
for bar, val in zip(bars2, recalls):                  # перебираем столбцы
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,  # позиция текста
             f'{val:.2f}', ha='center', fontsize=11)  # значение

plt.suptitle('Метрики оценки RAG-пайплайна', fontsize=16, fontweight='bold')  # общий заголовок
plt.tight_layout()                                    # подстройка отступов
plt.savefig('rag_evaluation.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация метрик оценки готова!")

---
## Шаг 11. Продвинутый: ре-ранжирование и гибридный поиск

Семантический поиск хорош, но иногда **ключевые слова** работают лучше для точных совпадений. Решение - **гибридный поиск**!

### Идея гибридного поиска:
1. **BM25** ищет по ключевым словам (лексический поиск)
2. **Семантический** ищет по смыслу (векторный поиск)
3. Объединяем результаты с **взвешенной суммой** или **ре-ранжированием**

### Ре-ранжирование:
После первоначального поиска, мы можем **переупорядочить** результаты с помощью более точной модели (cross-encoder).

In [ ]:
# ============================================================
# ШАГ 11: BM25 - лексический поиск по ключевым словам
# ============================================================

from rank_bm25 import BM25Okapi                       # импортируем BM25

# Подготавливаем корпус для BM25
corpus = rag_documents                                # документы для поиска
tokenized_corpus = [doc.lower().split() for doc in corpus]  # токенизируем (простая разбивка)

# Создаём BM25 индекс
bm25 = BM25Okapi(tokenized_corpus)                    # строим BM25 индекс

# Функция поиска BM25
def search_bm25(query, bm25_index, documents, top_k=3):
    # Ищем документы с помощью BM25
    tokenized_query = query.lower().split()           # токенизируем запрос
    scores = bm25_index.get_scores(tokenized_query)   # получаем оценки BM25
    top_indices = np.argsort(scores)[::-1][:top_k]    # индексы top-k по убыванию
    results = []                                      # список результатов
    for idx in top_indices:                           # перебираем индексы
        results.append({                              # добавляем результат
            "document": documents[idx],               # текст документа
            "score": float(scores[idx]),              # оценка BM25
            "index": int(idx)                         # индекс
        })
    return results                                    # возвращаем результаты

# Тестируем BM25
print("BM25 - Лексический поиск по ключевым словам")
print("=" * 60)                                       # разделитель

bm25_test_queries = [                                 # тестовые запросы
    "нейронные сети",                                 # запрос 1
    "RAG поиск генерация",                            # запрос 2
]

for query in bm25_test_queries:                       # перебираем запросы
    results = search_bm25(query, bm25, corpus, top_k=3)  # ищем
    print(f"\nЗапрос: '{query}'")                    # выводим запрос
    for r in results:                                 # перебираем результаты
        print(f"  [BM25={r['score']:.4f}] {r['document'][:70]}...")  # выводим

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Гибридный поиск - BM25 + семантический
# ============================================================

def hybrid_search(query, model, faiss_index, bm25_index, documents, top_k=5, alpha=0.5):
    # Гибридный поиск: комбинация BM25 и семантического поиска
    # alpha: вес семантического поиска (0 = только BM25, 1 = только семантический)

    # 1. Семантический поиск (FAISS)
    query_emb = model.encode([query], normalize_embeddings=True)  # эмбеддинг запроса
    sem_scores, sem_indices = faiss_index.search(query_emb.astype(np.float32), top_k)  # поиск
    semantic_results = {}                              # словарь семантических результатов
    for score, idx in zip(sem_scores[0], sem_indices[0]):  # перебираем
        if idx >= 0:                                  # если валидный
            semantic_results[int(idx)] = float(score)  # сохраняем оценку

    # 2. Лексический поиск (BM25)
    tokenized_query = query.lower().split()           # токенизируем запрос
    bm25_scores = bm25_index.get_scores(tokenized_query)  # оценки BM25
    bm25_top = np.argsort(bm25_scores)[::-1][:top_k]  # top-k BM25
    bm25_results = {}                                 # словарь BM25 результатов
    for idx in bm25_top:                              # перебираем
        bm25_results[int(idx)] = float(bm25_scores[idx])  # сохраняем

    # 3. Нормализуем оценки в диапазон [0, 1]
    def normalize_scores(scores_dict):                # функция нормализации
        if not scores_dict:                           # если пусто
            return {}                                 # возвращаем пустой
        values = list(scores_dict.values())           # значения
        min_val, max_val = min(values), max(values)   # минимум и максимум
        if max_val == min_val:                        # если все одинаковые
            return {k: 1.0 for k in scores_dict}     # все = 1
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores_dict.items()}  # нормализуем

    norm_sem = normalize_scores(semantic_results)     # нормализуем семантические
    norm_bm25 = normalize_scores(bm25_results)        # нормализуем BM25

    # 4. Объединяем с весом alpha
    all_indices = set(list(norm_sem.keys()) + list(norm_bm25.keys()))  # все индексы
    combined = {}                                     # объединённые оценки
    for idx in all_indices:                           # перебираем
        sem_score = norm_sem.get(idx, 0.0)            # семантическая оценка
        bm25_score = norm_bm25.get(idx, 0.0)          # BM25 оценка
        combined[idx] = alpha * sem_score + (1 - alpha) * bm25_score  # взвешенная сумма

    # 5. Сортируем по объединённой оценке
    sorted_results = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]  # top-k
    results = []                                      # список результатов
    for idx, score in sorted_results:                 # перебираем
        results.append({                              # добавляем результат
            "document": documents[idx],               # текст документа
            "combined_score": score,                  # объединённая оценка
            "semantic_score": semantic_results.get(idx, 0.0),  # семантическая
            "bm25_score": bm25_results.get(idx, 0.0),  # BM25
            "index": idx                              # индекс
        })
    return results                                    # возвращаем результаты

# Тестируем гибридный поиск
print("ГИБРИДНЫЙ ПОИСК: BM25 + Семантический")
print("=" * 70)                                       # разделитель

test_query = "как работает RAG поиск"                 # тестовый запрос

# Сравниваем: только BM25, только семантический, гибридный
print(f"\nЗапрос: '{test_query}'")                   # выводим запрос

for alpha, name in [(0.0, "Только BM25"), (1.0, "Только семантический"), (0.5, "Гибридный (50/50)")]:  # перебираем
    results = hybrid_search(test_query, model, index, bm25, rag_documents,  # ищем
                           top_k=3, alpha=alpha)       # параметры поиска
    print(f"\n{name}:")                              # название подхода
    for r in results:                                 # перебираем результаты
        print(f"  [combined={r['combined_score']:.4f}, sem={r['semantic_score']:.4f}, bm25={r['bm25_score']:.4f}] {r['document'][:60]}...")  # результат

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Ре-ранжирование результатов
# ============================================================

def rerank_results(query, initial_results, model, top_k=3):
    # Ре-ранжирование: оцениваем каждую пару (запрос, документ) более точно
    # Используем косинусное сходство эмбеддингов как скорер
    query_emb = model.encode([query], normalize_embeddings=True)  # эмбеддинг запроса

    reranked = []                                     # список ре-ранжированных результатов
    for result in initial_results:                    # перебираем начальные результаты
        doc_emb = model.encode([result['document']], normalize_embeddings=True)  # эмбеддинг документа
        # Вычисляем точное косинусное сходство
        score = cosine_similarity(query_emb, doc_emb)[0][0]  # точная оценка
        reranked.append({                             # добавляем результат
            "document": result['document'],           # текст
            "original_score": result.get('combined_score', result.get('score', 0)),  # оригинальная оценка
            "rerank_score": float(score),             # оценка после ре-ранжирования
            "index": result.get('index', -1)          # индекс
        })

    # Сортируем по ре-ранжированной оценке
    reranked.sort(key=lambda x: x['rerank_score'], reverse=True)  # сортируем
    return reranked[:top_k]                           # возвращаем top-k

# Получаем начальные результаты и ре-ранжируем
initial_results = hybrid_search(test_query, model, index, bm25, rag_documents,  # начальный поиск
                               top_k=5, alpha=0.5)    # параметры

reranked = rerank_results(test_query, initial_results, model, top_k=3)  # ре-ранжируем

print("РЕ-РАНЖИРОВАНИЕ РЕЗУЛЬТАТОВ")
print("=" * 70)                                       # разделитель
print(f"Запрос: '{test_query}'")                     # выводим запрос

print("\nДо ре-ранжирования:")                       # заголовок
for i, r in enumerate(initial_results[:3], 1):        # перебираем начальные
    print(f"  {i}. [score={r.get('combined_score', r.get('score', 0)):.4f}] {r['document'][:60]}...")  # результат

print("\nПосле ре-ранжирования:")                     # заголовок
for i, r in enumerate(reranked, 1):                   # перебираем ре-ранжированные
    print(f"  {i}. [rerank={r['rerank_score']:.4f}, original={r['original_score']:.4f}] {r['document'][:60]}...")  # результат

print("\nРе-ранжирование может изменить порядок результатов!")  # вывод

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Визуализация гибридного поиска
# ============================================================

# Сравниваем качество трёх подходов поиска

# Создаём тестовый набор для сравнения
comparison_queries = [                                # запросы для сравнения
    "Что такое нейронные сети?",                      # запрос 1
    "Как работает RAG?",                              # запрос 2
    "FAISS векторный поиск",                          # запрос 3
    "BERT и GPT трансформеры",                        # запрос 4
    "Стратегии чанкинга документов",                  # запрос 5
]

# Релевантные документы для каждого запроса (индексы)
relevant_per_query = [                                # релевантные индексы
    [2],                                              # для запроса 1
    [8, 9],                                           # для запроса 2
    [10, 7],                                          # для запроса 3
    [6, 5],                                           # для запроса 4
    [9, 13],                                          # для запроса 5
]

# Оцениваем три подхода
approaches = {                                        # подходы к поиску
    "BM25": [],                                       # только BM25
    "Семантический": [],                              # только семантический
    "Гибридный": [],                                  # гибридный
}

for query, relevant in zip(comparison_queries, relevant_per_query):  # перебираем запросы
    # BM25
    bm25_res = search_bm25(query, bm25, rag_documents, top_k=3)  # BM25 поиск
    bm25_relevant = [1 if r['index'] in relevant else 0 for r in bm25_res]  # релевантность

    # Семантический
    sem_res = search_faiss(query, model, index, rag_documents, top_k=3)  # семантический
    sem_relevant = [1 if r['index'] in relevant else 0 for r in sem_res]  # релевантность

    # Гибридный
    hybrid_res = hybrid_search(query, model, index, bm25, rag_documents, top_k=3, alpha=0.5)  # гибридный
    hybrid_relevant = [1 if r['index'] in relevant else 0 for r in hybrid_res]  # релевантность

    approaches["BM25"].append(precision_at_k(bm25_relevant, 3))  # precision BM25
    approaches["Семантический"].append(precision_at_k(sem_relevant, 3))  # precision семантический
    approaches["Гибридный"].append(precision_at_k(hybrid_relevant, 3))  # precision гибридный

# Визуализируем сравнение
fig, ax = plt.subplots(figsize=(10, 6))               # создаём фигуру

x = np.arange(len(comparison_queries))                # позиции по X
width = 0.25                                          # ширина столбцов

colors = ['#e74c3c', '#3498db', '#2ecc71']            # цвета подходов
for i, (name, scores) in enumerate(approaches.items()):  # перебираем подходы
    ax.bar(x + i*width, scores, width, label=name, color=colors[i], alpha=0.8, edgecolor='black')  # столбцы

ax.set_xlabel('Запрос', fontsize=13)                  # подпись X
ax.set_ylabel('Precision@3', fontsize=13)             # подпись Y
ax.set_title('Сравнение подходов поиска: BM25 vs Семантический vs Гибридный', fontsize=14, fontweight='bold')  # заголовок
ax.set_xticks(x + width)                             # деления X
ax.set_xticklabels([q[:20] + "..." for q in comparison_queries], rotation=30, ha='right', fontsize=9)  # подписи X
ax.legend(fontsize=11)                                # легенда
ax.set_ylim(0, 1.2)                                  # пределы Y

plt.tight_layout()                                    # подстройка отступов
plt.savefig('hybrid_search_comparison.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Сравнение подходов поиска готово!")

---
## Шаг 12. RAG vs fine-tuning vs prompting - что выбрать?

Три подхода к адаптации LLM под конкретную задачу. У каждого свои плюсы и минусы.

### Сравнение

| Критерий | RAG | Fine-tuning | Prompting |
|----------|-----|-------------|-----------|
| **Стоимость** | Низкая (инфраструктура поиска) | Высокая (GPU, данные) | Минимальная |
| **Свежесть данных** | Реальное время | Периодическое переобучение | Зависит от промпта |
| **Объяснимость** | Высокая (источники) | Низкая | Средняя |
| **Сложность внедрения** | Средняя | Высокая | Низкая |
| **Качество для специфичных задач** | Среднее | Высокое | Низкое |
| **Затраты на поддержку** | Низкие | Высокие | Минимальные |

### Когда что использовать?

- **RAG**: нужны актуальные данные, объяснимость, частые обновления
- **Fine-tuning**: нужна специализация, другой стиль/тон, уникальная задача
- **Prompting**: быстрый прототип, простые задачи, нет бюджета

In [ ]:
# ============================================================
# ШАГ 12: Визуализация сравнения RAG vs Fine-tuning vs Prompting
# ============================================================

# Создаём радарную диаграмму для сравнения трёх подходов

categories = ['Стоимость', 'Свежесть\nданных', 'Объяснимость',  # категории для сравнения
              'Качество', 'Простота\nвнедрения', 'Масштабируемость']

# Оценки от 1 до 10 для каждого подхода
rag_scores = [7, 9, 9, 7, 6, 8]                      # RAG
finetune_scores = [3, 3, 4, 9, 3, 5]                 # Fine-tuning
prompting_scores = [9, 5, 6, 4, 9, 7]                # Prompting

# Создаём радарную диаграмму
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))  # полярная система координат

# Углы для каждой категории
N = len(categories)                                   # количество категорий
angles = [n / float(N) * 2 * math.pi for n in range(N)]  # углы
angles += angles[:1]                                  # замыкаем многоугольник

# Добавляем оценки (замыкаем)
rag_values = rag_scores + rag_scores[:1]              # замыкаем RAG
finetune_values = finetune_scores + finetune_scores[:1]  # замыкаем fine-tuning
prompting_values = prompting_scores + prompting_scores[:1]  # замыкаем prompting

# Рисуем многоугольники
ax.plot(angles, rag_values, 'o-', linewidth=2, label='RAG', color='#2ecc71')  # RAG
ax.fill(angles, rag_values, alpha=0.15, color='#2ecc71')  # заливка RAG
ax.plot(angles, finetune_values, 's-', linewidth=2, label='Fine-tuning', color='#e74c3c')  # Fine-tuning
ax.fill(angles, finetune_values, alpha=0.15, color='#e74c3c')  # заливка FT
ax.plot(angles, prompting_values, '^-', linewidth=2, label='Prompting', color='#3498db')  # Prompting
ax.fill(angles, prompting_values, alpha=0.15, color='#3498db')  # заливка Prompting

# Настройка осей
ax.set_xticks(angles[:-1])                           # деления
ax.set_xticklabels(categories, fontsize=12)           # подписи
ax.set_ylim(0, 10)                                   # пределы
ax.set_yticks([2, 4, 6, 8, 10])                     # деления Y
ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=9)  # подписи Y
ax.set_title('RAG vs Fine-tuning vs Prompting', fontsize=16, fontweight='bold', pad=20)  # заголовок
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)  # легенда

plt.tight_layout()                                    # подстройка отступов
plt.savefig('rag_vs_finetuning_vs_prompting.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Радарная диаграмма сравнения готова!")

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Когда использовать каждый подход
# ============================================================

# Система рекомендаций: какой подход выбрать

def recommend_approach(needs):
    # Рекомендуем подход на основе потребностей пользователя
    scores = {"RAG": 0, "Fine-tuning": 0, "Prompting": 0}  # начальные баллы

    for need, weight in needs.items():                # перебираем потребности
        if need == "актуальные_данные":              # нужны свежие данные
            scores["RAG"] += 3 * weight              # RAG лучший
            scores["Fine-tuning"] += 1 * weight      # FT хуже
            scores["Prompting"] += 1 * weight        # prompting хуже
        elif need == "объяснимость":                  # нужны источники
            scores["RAG"] += 3 * weight              # RAG лучший
            scores["Fine-tuning"] += 1 * weight
            scores["Prompting"] += 2 * weight
        elif need == "специфичный_стиль":            # свой стиль/тон
            scores["RAG"] += 1 * weight
            scores["Fine-tuning"] += 3 * weight      # FT лучший
            scores["Prompting"] += 2 * weight
        elif need == "низкая_стоимость":              # дёшево
            scores["RAG"] += 2 * weight
            scores["Fine-tuning"] += 0 * weight
            scores["Prompting"] += 3 * weight        # prompting лучший
        elif need == "быстрый_старт":                 # быстро запустить
            scores["RAG"] += 2 * weight
            scores["Fine-tuning"] += 0 * weight
            scores["Prompting"] += 3 * weight        # prompting лучший
        elif need == "высокое_качество":              # высокое качество
            scores["RAG"] += 2 * weight
            scores["Fine-tuning"] += 3 * weight      # FT лучший
            scores["Prompting"] += 1 * weight

    # Сортируем по баллам
    recommendation = sorted(scores.items(), key=lambda x: x[1], reverse=True)  # сортируем
    return recommendation                              # возвращаем рекомендацию

# Примеры сценариев
scenarios = [                                         # разные сценарии
    {                                                 # сценарий 1
        "name": "Корпоративный чат-бот с документацией",
        "needs": {"актуальные_данные": 3, "объяснимость": 3, "низкая_стоимость": 2}
    },
    {                                                 # сценарий 2
        "name": "Медицинский ассистент со своим стилем",
        "needs": {"специфичный_стиль": 3, "высокое_качество": 3, "объяснимость": 2}
    },
    {                                                 # сценарий 3
        "name": "Быстрый прототип для демо",
        "needs": {"быстрый_старт": 3, "низкая_стоимость": 3, "актуальные_данные": 1}
    },
]

print("РЕКОМЕНДАЦИИ ПО ВЫБОРУ ПОДХОДА")
print("=" * 70)                                       # разделитель

for scenario in scenarios:                            # перебираем сценарии
    print(f"\nСценарий: {scenario['name']}")         # название
    rec = recommend_approach(scenario['needs'])        # рекомендация
    for approach, score in rec:                       # перебираем результаты
        bar = '#' * int(score * 3)                    # визуальная полоска
        print(f"  {approach:15s}: {score:5.1f} {bar}")  # выводим
    print(f"  -> Рекомендация: {rec[0][0]}")          # лучшая рекомендация

---
## Шаг 13. Практические задания

Теперь ваша очередь! Примените знания, полученные в этом ноутбуке.

---

### Задание 1: Своя база знаний
Создайте RAG-систему с 10+ документами на тему, которая вам интересна (спорт, кулинария, программирование и т.д.). Добавьте документы, создайте индекс и протестируйте 5 запросов.

**Подсказка:** Используйте класс `SimpleRAG` и метод `add_documents()`.

---

### Задание 2: Оптимизация чанкинга
Исследуйте, как размер чанка влияет на качество поиска:
1. Создайте RAG с `chunk_size = 50, 100, 200, 400, 800`
2. Для каждого размера измерьте precision@3 и recall@3
3. Постройте график зависимости качества от размера чанка
4. Найдите оптимальный размер

**Подсказка:** Используйте функции `precision_at_k()` и `recall_at_k()`.

---

### Задание 3: Гибридный поиск с настройкой alpha
Исследуйте влияние веса alpha на качество гибридного поиска:
1. Протестируйте alpha = 0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0
2. Для каждого alpha измерьте precision@3
3. Найдите оптимальный alpha
4. Объясните, почему он оптимален

**Подсказка:** Используйте функцию `hybrid_search()` с разными alpha.

---

### Задание 4: Расширенный FastAPI сервер
Добавьте в RAG-сервер новые эндпоинты:
1. `/health` - проверка здоровья сервера
2. `/documents` - список всех документов
3. `/delete_document/{index}` - удаление документа
4. `/evaluate` - оценка качества на тестовом наборе

**Подсказка:** Используйте декораторы `@app.get()` и `@app.delete()`.

---

### Задание 5: Мультиязычный RAG
Создайте RAG-систему, работающую с русскими и английскими документами:
1. Используйте модель `paraphrase-multilingual-MiniLM-L12-v2`
2. Добавьте документы на обоих языках
3. Протестируйте кросс-язычный поиск (запрос на русском, документы на английском)
4. Сравните качество с одноязычной моделью

**Подсказка:** Мультиязычная модель создаёт общее пространство для разных языков.

---

### Критерии оценки
| Задание | Базовый уровень | Продвинутый уровень |
|---------|----------------|-------------------|
| 1 | 10+ документов, 5 запросов | + визуализация результатов |
| 2 | График зависимости | + объяснение оптимального размера |
| 3 | Таблица alpha vs качество | + объяснение и визуализация |
| 4 | 2+ новых эндпоинта | + полный CRUD + тесты |
| 5 | Кросс-язычный поиск | + сравнительный анализ |

---

## Итоги

Поздравляем! Вы изучили RAG (Retrieval-Augmented Generation) от основ до продвинутых техник:

### Что мы сделали:
1. Поняли, **почему** LLM галлюцинируют и как RAG решает проблему
2. Разобрали **архитектуру RAG**: retrieve -> generate
3. Реализовали **три стратегии чанкинга** и сравнили их
4. Создали **эмбеддинги** с помощью sentence-transformers
5. Построили **FAISS индекс** с нуля и освоили поиск
6. Собрали **полный RAG-пайплайн** от загрузки до генерации
7. Запустили **FastAPI сервер** с CRUD эндпоинтами
8. Исследовали RAG через **интерактивные виджеты**
9. Оценили качество с помощью **precision@k, recall, MRR**
10. Реализовали **гибридный поиск** (BM25 + семантический)
11. Сравнили **RAG vs fine-tuning vs prompting**

### Ключевые выводы:
- RAG - самый практичный способ дать LLM доступ к актуальным данным
- Чанкинг и выбор модели эмбеддингов сильно влияют на качество
- Гибридный поиск обычно лучше, чем только семантический или только BM25
- Оценка качества - обязательный этап внедрения RAG
- Выбор между RAG, fine-tuning и prompting зависит от задачи

**Продолжайте экспериментировать!**